# 03 — Vector index: reviews → MiniLM embeddings → Weaviate

Pipeline B (unstructured text). Each review with a title or text becomes one document
(`Title. Review text`), embedded locally with `all-MiniLM-L6-v2` (384 numbers) and stored in
Weaviate Cloud with its metadata (rating, department, …).

Search is **hybrid**: BM25 keyword search + vector search with the same filters, fused with
**Reciprocal Rank Fusion** in our own code. `review_id` links every result back to SQLite.

| Section | Builds |
|---|---|
| 0 | Setup |
| 1 | Load review documents from SQLite (read-only) |
| 2 | Embeddings, cached to disk |
| 3 | Connect to Weaviate, create the collection |
| 4 | Upload |
| 5 | Filters, keyword search, semantic search, RRF |
| 6 | Checks |

## Section 0 — Setup


In [1]:
import hashlib
import os
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd
import weaviate
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from weaviate.classes.config import Configure, DataType, Property, Tokenization
from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery
from weaviate.util import generate_uuid5


d:\ChatBot test\.retailbot\Lib\site-packages\authlib\integrations\httpx_client\assertion_client.py:5: AuthlibDeprecationWarning: The httpx module is deprecated; please use httpx2 instead.
  from ._compat import httpx2
d:\ChatBot test\.retailbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "processed" / "chatbot.db"
EMBEDDINGS_FILE = PROJECT_ROOT / "data" / "processed" / "review_embeddings.npz"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "Review"

load_dotenv(PROJECT_ROOT / ".env")
missing = [k for k in ("WEAVIATE_URL", "WEAVIATE_ADMIN_KEY", "WEAVIATE_API_KEY") if not os.getenv(k)]
print("DB exists:", DB_PATH.exists(), "| missing .env keys:", missing or "none")


DB exists: True | missing .env keys: none


## Section 1 — Load review documents from SQLite
Read-only connection. One document per review: `Title. Review text`.
Reviews with neither (844) are skipped — there is nothing to search.


In [3]:
def connect_db_read_only(path):
    conn = sqlite3.connect(f"{Path(path).resolve().as_uri()}?mode=ro", uri=True)
    conn.row_factory = sqlite3.Row
    return conn


def none_if_missing(value):
    """SQL NULL arrives in a DataFrame as NaN/None; Weaviate wants None."""
    return None if pd.isna(value) else value


def to_document(title, text):
    """'Title. Review text' - skipping blank parts."""
    return ". ".join(part for part in (title, text) if isinstance(part, str) and part)


def load_review_documents(db_path, limit=None):
    conn = connect_db_read_only(db_path)
    try:
        sql = """
            SELECT review_id, _row_id AS source_row, clothing_id, age, title, review_text, rating,
                   recommended_ind, positive_feedback_count, division_name, department_name, class_name
            FROM reviews
            WHERE title IS NOT NULL OR review_text IS NOT NULL
            ORDER BY review_id"""
        if limit:
            sql += f" LIMIT {int(limit)}"
        reviews = pd.DataFrame([dict(r) for r in conn.execute(sql)])
    finally:
        conn.close()
    reviews["document"] = [to_document(t, x) for t, x in zip(reviews["title"], reviews["review_text"])]
    return reviews


reviews = load_review_documents(DB_PATH)
print(len(reviews), "documents | empty:", (reviews["document"] == "").sum())
reviews[["review_id", "rating", "department_name", "document"]].head(3)


22642 documents | empty: 0


,review_id,rating,department_name,document
0,0,4,Intimate,Absolutely wonderful - silky and sexy and comf...
1,1,5,Dresses,Love this dress! it's sooo pretty. i happene...
2,2,3,Dresses,Some major design flaws. I had such high hopes...


## Section 2 — Embeddings (cached)
MiniLM turns each document into 384 numbers; similar meanings → nearby vectors.
On CPU this takes ~12 minutes for 22,642 reviews, so vectors are saved to
`review_embeddings.npz` and reused. A **fingerprint** of all IDs + texts detects stale caches:
if the data changes, the cache is recomputed automatically.


In [4]:
def documents_fingerprint(review_ids, documents):
    """Changes whenever any review ID or text changes - used to decide if the cache is stale."""
    digest = hashlib.sha256()
    for review_id, document in zip(review_ids, documents):
        digest.update(f"{review_id}\x1f{document}\x1e".encode("utf-8"))
    return digest.hexdigest()


def load_or_compute_embeddings(reviews, model, cache_file=EMBEDDINGS_FILE):
    fingerprint = documents_fingerprint(reviews["review_id"], reviews["document"])
    if cache_file.exists():
        cached = np.load(cache_file, allow_pickle=False)
        if str(cached["fingerprint"]) == fingerprint and str(cached["model"]) == MODEL_NAME:
            print(f"Loaded {len(cached['vectors'])} cached vectors from {cache_file.name}")
            return cached["vectors"]
        print("Cache is stale (data or model changed) - recomputing")
    start = time.time()
    vectors = model.encode(reviews["document"].tolist(), batch_size=64,
                           normalize_embeddings=True, show_progress_bar=True)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    np.savez(cache_file, vectors=vectors, review_ids=reviews["review_id"].to_numpy(),
             fingerprint=fingerprint, model=MODEL_NAME)
    print(f"Embedded {len(vectors)} documents in {time.time() - start:.0f}s -> saved {cache_file.name}")
    return vectors


In [5]:
model = SentenceTransformer(MODEL_NAME)
print("Vector size:", model.get_embedding_dimension())
vectors = load_or_compute_embeddings(reviews, model)
print(vectors.shape, vectors.dtype)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 935.94it/s]


Vector size: 384


Batches: 100%|██████████| 354/354 [17:34<00:00,  2.98s/it]


Embedded 22642 documents in 1055s -> saved review_embeddings.npz
(22642, 384) float32


## Section 3 — Connect to Weaviate and create the collection
Two keys: **admin** (create + upload, used only here) and **viewer** (search only, used by the chatbot).
The collection is like a table: each review is one object with a vector + properties.


In [6]:
def connect_weaviate(admin=False):
    """admin=True only for building the index; the chatbot searches with the read-only viewer key."""
    key = os.environ["WEAVIATE_ADMIN_KEY" if admin else "WEAVIATE_API_KEY"]
    return weaviate.connect_to_weaviate_cloud(
        cluster_url=os.environ["WEAVIATE_URL"],
        auth_credentials=Auth.api_key(key),
    )


In [7]:
CATEGORY = Tokenization.FIELD   # whole value is one token: exact filters, not word search

REVIEW_PROPERTIES = [
    Property(name="review_id", data_type=DataType.INT),
    Property(name="source_row", data_type=DataType.INT),
    Property(name="clothing_id", data_type=DataType.INT),
    Property(name="age", data_type=DataType.INT),
    Property(name="rating", data_type=DataType.INT),
    Property(name="recommended_ind", data_type=DataType.INT),
    Property(name="positive_feedback_count", data_type=DataType.INT),
    Property(name="title", data_type=DataType.TEXT),
    Property(name="review_text", data_type=DataType.TEXT),
    Property(name="division_name", data_type=DataType.TEXT, tokenization=CATEGORY),
    Property(name="department_name", data_type=DataType.TEXT, tokenization=CATEGORY),
    Property(name="class_name", data_type=DataType.TEXT, tokenization=CATEGORY),
]


def create_review_collection(client, name=COLLECTION_NAME, replace=False):
    if client.collections.exists(name):
        if not replace:
            raise ValueError(f"Collection '{name}' already exists. Use replace=True to rebuild it.")
        client.collections.delete(name)
    return client.collections.create(
        name=name,
        vector_config=Configure.Vectors.self_provided(),                       # we bring MiniLM vectors
        inverted_index_config=Configure.inverted_index(index_null_state=True),  # allow "is blank" filters
        properties=REVIEW_PROPERTIES,
    )


## Section 4 — Upload
Batches of 200. Each object's ID is derived from its `review_id`, so re-running the upload
updates objects instead of duplicating them. Weaviate Cloud indexes vectors in the background,
so we wait until indexing finishes before counting or searching.


In [8]:
PROPERTY_NAMES = [p.name for p in REVIEW_PROPERTIES]


def review_properties(row):
    return {name: none_if_missing(row[name]) for name in PROPERTY_NAMES}


def upload_reviews(client, collection, reviews, vectors, batch_size=200):
    start = time.time()
    with collection.batch.fixed_size(batch_size=batch_size) as batch:
        for row, vector in zip(reviews.to_dict("records"), vectors):
            batch.add_object(
                properties=review_properties(row),
                vector=vector.tolist(),
                uuid=generate_uuid5(int(row["review_id"])),   # same review -> same ID -> no duplicates
            )
    failed = collection.batch.failed_objects
    client.batch.wait_for_vector_indexing()                     # Weaviate Cloud indexes asynchronously
    total = collection.aggregate.over_all(total_count=True).total_count
    print(f"Uploaded {len(reviews)} in {time.time() - start:.0f}s | failed: {len(failed)} | in collection: {total}")
    if failed:
        print("First failure:", failed[0].message)
    return total


In [9]:
admin = connect_weaviate(admin=True)
try:
    reviews_collection = create_review_collection(admin, replace=True)
    total = upload_reviews(admin, reviews_collection, reviews, vectors)
finally:
    admin.close()


Uploaded 22642 in 41s | failed: 0 | in collection: 22642


### Check: the index matches SQLite


In [10]:
admin = connect_weaviate(admin=True)
try:
    collection = admin.collections.get(COLLECTION_NAME)

    # 1. Every document arrived
    total = collection.aggregate.over_all(total_count=True).total_count
    assert total == len(reviews), (total, len(reviews))

    # 2. Metadata matches SQLite for a known review
    obj = collection.query.fetch_object_by_id(generate_uuid5(2))
    assert obj.properties["title"] == "Some major design flaws"
    assert (obj.properties["rating"], obj.properties["source_row"]) == (3, 4)

    # 3. Blank departments are stored as blank, and can be filtered
    blank = collection.aggregate.over_all(
        filters=Filter.by_property("department_name").is_none(True), total_count=True).total_count
    assert blank == reviews["department_name"].isna().sum(), blank

    # 4. Self-retrieval: each stored vector finds its own review first
    for i in (0, 5000, 22641):
        hit = collection.query.near_vector(near_vector=vectors[i].tolist(), limit=1).objects[0]
        assert hit.properties["review_id"] == reviews["review_id"].iloc[i]

    print(f"All 4 checks passed: {total} reviews indexed, {blank} with no department")
finally:
    admin.close()


All 4 checks passed: 22642 reviews indexed, 13 with no department


In [11]:
viewer = connect_weaviate(admin=False)
try:
    collection = viewer.collections.get(COLLECTION_NAME)
    print("Viewer can search:", len(collection.query.bm25(query="dress", limit=3).objects), "results")
    try:
        collection.data.delete_by_id(generate_uuid5(0))
        print("UNEXPECTED: viewer could delete")
    except Exception as exc:
        print("Viewer write blocked as expected ->", type(exc).__name__)
finally:
    viewer.close()


Viewer can search: 3 results
Viewer write blocked as expected -> InsufficientPermissionsError


## Section 5 — Search: filters, keyword, semantic, RRF
`build_filter()` turns a small dict into a Weaviate filter and rejects anything unknown —
later the LLM will write these dicts. Both searches use the same filter; RRF merges their rankings.


In [12]:
FILTERABLE = {
    "review_id": int, "clothing_id": int, "age": int, "rating": int, "recommended_ind": int,
    "positive_feedback_count": int, "division_name": str, "department_name": str, "class_name": str,
}


def build_filter(conditions):
    """{"department_name": "Tops", "rating": {"<=": 2}, "class_name": {"in": [...]}} -> Weaviate Filter.

    Only known properties and operators are allowed: later the LLM will produce these dicts.
    """
    if not conditions:
        return None
    parts = []
    for name, condition in conditions.items():
        if name not in FILTERABLE:
            raise ValueError(f"Cannot filter on '{name}'. Allowed: {sorted(FILTERABLE)}")
        if not isinstance(condition, dict):
            condition = {"=": condition}
        prop = Filter.by_property(name)
        for op, value in condition.items():
            if op == "in":
                parts.append(prop.contains_any([FILTERABLE[name](v) for v in value]))
            elif op == "is_blank":
                parts.append(prop.is_none(bool(value)))
            elif op in ("=", "!=", "<", "<=", ">", ">="):
                value = FILTERABLE[name](value)
                parts.append({
                    "=": prop.equal, "!=": prop.not_equal, "<": prop.less_than,
                    "<=": prop.less_or_equal, ">": prop.greater_than, ">=": prop.greater_or_equal,
                }[op](value))
            else:
                raise ValueError(f"Unknown operator '{op}' for '{name}'")
    return Filter.all_of(parts)


In [13]:
SEARCH_FIELDS = ["title", "review_text"]


def semantic_search(collection, model, query, filters=None, limit=50):
    vector = model.encode(query, normalize_embeddings=True).tolist()
    result = collection.query.near_vector(near_vector=vector, limit=limit, filters=filters,
                                          return_metadata=MetadataQuery(distance=True))
    return result.objects


def keyword_search(collection, query, filters=None, limit=50):
    result = collection.query.bm25(query=query, query_properties=SEARCH_FIELDS, limit=limit,
                                   filters=filters, return_metadata=MetadataQuery(score=True))
    return result.objects


In [14]:
def rrf_fuse(rankings, k=60):
    """Reciprocal Rank Fusion: score(d) = sum over rankings of 1 / (k + rank of d)."""
    scores = {}
    for ranking in rankings:
        for rank, review_id in enumerate(ranking, start=1):
            scores[review_id] = scores.get(review_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)


def hybrid_search(collection, model, query, conditions=None, top_k=10, candidates=50, rrf_k=60):
    """Keyword + semantic search with the same filters, fused with RRF. Returns a DataFrame."""
    filters = build_filter(conditions)
    semantic = semantic_search(collection, model, query, filters, candidates)
    keyword = keyword_search(collection, query, filters, candidates)

    by_id = {o.properties["review_id"]: o.properties for o in semantic + keyword}
    semantic_rank = {o.properties["review_id"]: r for r, o in enumerate(semantic, start=1)}
    keyword_rank = {o.properties["review_id"]: r for r, o in enumerate(keyword, start=1)}

    rows = []
    for review_id, score in rrf_fuse([list(semantic_rank), list(keyword_rank)], k=rrf_k)[:top_k]:
        props = by_id[review_id]
        rows.append({
            "review_id": review_id,
            "rrf_score": round(score, 4),
            "semantic_rank": semantic_rank.get(review_id),
            "keyword_rank": keyword_rank.get(review_id),
            "rating": props["rating"],
            "department_name": props["department_name"],
            "class_name": props["class_name"],
            "source_row": props["source_row"],
            "text": to_document(props["title"], props["review_text"]),
        })
    return pd.DataFrame(rows)


## Section 6 — Checks
Searches use the **viewer** key: the same read-only access the chatbot will have.


In [15]:
# RRF on the worked example from the learning doc
fused = rrf_fuse([["a", "b", "c"], ["c", "a", "d"]])
assert [d for d, _ in fused] == ["a", "c", "b", "d"]
assert round(fused[0][1], 4) == round(1 / 61 + 1 / 62, 4)
print("rrf_fuse: OK", [(d, round(s, 4)) for d, s in fused])

# The filter builder rejects what it doesn't know
for bad in [{"review_text": "x"}, {"rating": {"~": 2}}, {"rating": {"<=": "two"}}]:
    try:
        build_filter(bad)
        print("UNEXPECTED: accepted", bad)
    except (ValueError, TypeError) as exc:
        print("Rejected as expected ->", exc)


rrf_fuse: OK [('a', 0.0325), ('c', 0.0323), ('b', 0.0161), ('d', 0.0159)]
Rejected as expected -> Cannot filter on 'review_text'. Allowed: ['age', 'class_name', 'clothing_id', 'department_name', 'division_name', 'positive_feedback_count', 'rating', 'recommended_ind', 'review_id']
Rejected as expected -> Unknown operator '~' for 'rating'
Rejected as expected -> invalid literal for int() with base 10: 'two'


In [16]:
viewer = connect_weaviate(admin=False)
search_collection = viewer.collections.get(COLLECTION_NAME)

# Pretend the SQL step answered "Tops has the most negative reviews" -> it becomes the filter
tops_negative = {"department_name": "Tops", "rating": {"<=": 2}}
results = hybrid_search(search_collection, model, "runs small, too tight in the chest", tops_negative, top_k=5)

assert ((results["department_name"] == "Tops") & (results["rating"] <= 2)).all()
display(results.drop(columns="text"))
for text in results["text"].head(3):
    print("-", text[:150])


,review_id,rrf_score,semantic_rank,keyword_rank,rating,department_name,class_name,source_row
0,16541,0.0323,2,2,2,Tops,Fine gauge,16543
1,12138,0.0311,1,8,2,Tops,Knits,12140
2,23391,0.0307,10,1,2,Tops,Knits,23393
3,5907,0.0299,8,6,2,Tops,Knits,5909
4,8805,0.0294,6,10,2,Tops,Knits,8807


- Wanted to love but couldn't.. Runs small. arms very short and hits in the wrong place across your chest. would not recommend.
- Runs big. Pretty design and good quality but the medium was very balloony at the bottom. fit my chest alright (34c) but flared out way too much at the
- Tight in all the wrong places.... The model looks awesome in this cute top, but on me it looked boxy but was too tight around the chest and was too sh


In [17]:
filters = build_filter(tops_negative)
query = "runs small, too tight in the chest"
semantic_ids = [o.properties["review_id"] for o in semantic_search(search_collection, model, query, filters, 10)]
keyword_ids = [o.properties["review_id"] for o in keyword_search(search_collection, query, filters, 10)]
print("Semantic top 3:", semantic_ids[:3])
print("Keyword top 3: ", keyword_ids[:3])
print("Found by both in their top 10:", len(set(semantic_ids) & set(keyword_ids)))


Semantic top 3: [12138, 16541, 9000]
Keyword top 3:  [16541, 18117, 17236]
Found by both in their top 10: 3


In [18]:
see_through = hybrid_search(search_collection, model, "fabric is see-through",
                            {"department_name": "Dresses", "rating": {"<=": 2}}, top_k=5)
display(see_through.drop(columns="text"))
print("-", see_through["text"].iloc[0][:150])

no_department = hybrid_search(search_collection, model, "love it", {"department_name": {"is_blank": True}}, top_k=3)
print("Reviews with no department:", no_department["department_name"].tolist())

viewer.close()


,review_id,rrf_score,semantic_rank,keyword_rank,rating,department_name,class_name,source_row
0,9262,0.0328,1,1,1,Dresses,Dresses,9264
1,6911,0.0320,2,3,1,Dresses,Dresses,6913
2,17954,0.0318,4,2,2,Dresses,Dresses,17956
3,18831,0.0306,7,4,1,Dresses,Dresses,18833
4,11138,0.0288,12,7,2,Dresses,Dresses,11140


- See through!. I read all the reviews about the fabric being see through, but liked the dress enough to give it a shot. it is soooo see through. this t
Reviews with no department: [None, None, None]
